## Фундаментальные постулаты прикладной модели

Цель модели — не полное описание рынка, а получение исполнимого решения в момент $t$: открыть или закрыть позицию, пропустить вход либо признать состояние неопределённым.

Ниже зафиксированы рабочие допущения о рыночной структуре. Они не считаются универсальными законами. Каждый постулат должен проверяться на данных, а его нарушение должно приводить к разделению режимов, расширению модели или отказу от сделки.

### Обозначения

* $s(t)$ — направленный исполнимый L1-спред в момент $t$, отдельно для `long` и `short`;
* $\mathcal O_{\le t}$ — доступная к моменту $t$ история цен, объёмов, временных меток и признаков качества данных;
* $X_t$ — состояние рынка, вычисленное только из $\mathcal O_{\le t}$;
* $F_t$ — локальный равновесный уровень спреда, или флор;
* $t_{\mathrm{in}}$ — время срабатывания сигнала входа;
* $t_{\mathrm{out}}$ — время срабатывания сигнала выхода;
* $T_{\mathrm{in}}$ — задержка исполнения входа;
* $T_{\mathrm{out}}$ — задержка исполнения выхода;
* $q$ — номинал позиции;
* $\Pi$ — полный результат сделки после исполнения обеих ног входа и выхода.

В детерминированной бейзлайн-модели используется одна задержка:

$$
T_{\mathrm{in}}=T_{\mathrm{out}}=T.
$$

В стохастической модели задержки имеют общий закон распределения:

$$
T_{\mathrm{in}},T_{\mathrm{out}}\sim\mathcal D_T.
$$

Одинаковый закон распределения не означает автоматически независимость $T_{\mathrm{in}}$ и $T_{\mathrm{out}}$. Независимость является отдельным проверяемым допущением.

### P1. Рынок состоит из локальных статистических режимов

Динамика спреда неоднородна во времени, но допускает разделение на локальные режимы или кластеры с приблизительно устойчивыми статистическими свойствами.

Для сравнимых кластеров предполагается существование масштабирующего преобразования

$$
Z_c(u)=
\frac{s(t+\tau_c u)-F_c}{\sigma_c},
$$

при котором распределения нормированной динамики близки:

$$
\mathcal L(Z_c\mid c)\approx\mathcal P_*.
$$

Здесь $F_c$, $\sigma_c$ и $\tau_c$ задают характерный уровень, масштаб отклонений и временной масштаб кластера. Нормированное распределение $\mathcal P_*$ не обязано быть нормальным.

Сильное отклонение спреда может относиться к одному из разных режимов:

* временная дислокация с возвратом;
* переход к новому равновесному уровню;
* общий рыночный шок;
* артефакт или нарушение качества данных.

Поэтому большое значение спреда само по себе не означает возврат к прежнему флору. Возможность объединять кластеры должна подтверждаться на независимых монетах, днях и эпизодах.

Рост плотности тиков не считается автоматическим ростом числа независимых наблюдений.

### P2. Решение должно быть причинным

Любое состояние, параметр или решение стратегии в момент $t$ вычисляется только по информации, доступной не позднее $t$:

$$
X_t=\Phi(\mathcal O_{\le t}).
$$

Допускаются признаки памяти, включая взвешенные интегралы прошлых значений:

$$
M_t(h)=
\int_{-\infty}^{t}
K_h(t-u)\,g(\mathcal O_u)\,du,
$$

где $K_h$ — ядро памяти с масштабом $h$.

Выбор формы $K_h$, ширины памяти $h$ и состава $X_t$ является степенью свободы модели и должен фиксироваться до проверки на устойчивость.

Будущие значения не могут использоваться ни для расчёта признаков, ни для классификации текущего состояния.

### P3. В тихом режиме существует локальный флор

Для части рыночных режимов предполагается представление

$$
s(t)=F_t+\varepsilon_t,
$$

где $F_t$ изменяется медленнее характерного времени сделки, а шум $\varepsilon_t$ имеет робастный центр около нуля:

$$
Q_{50}^{\,w}(\varepsilon_t)\approx 0.
$$

Рабочая оценка флора строится по причинному тихому окну:

$$
\widehat F_t=
Q^{\,w}_{50}
\left(
s(u):u\in W_{\mathrm{quiet}}(t)
\right).
$$

Тихий диапазон определяется как

$$
\widehat{\mathcal F}_t=
\left[
Q^{\,w}_{25}(s(u)),
Q^{\,w}_{75}(s(u))
\right].
$$

Для позиции, открываемой на высоком положительном спреде, консервативной целью выхода может служить верхняя граница тихого диапазона:

$$
F_{\mathrm{exit}}=
Q^{\,w}_{75}(s(u)).
$$

Вес наблюдения должен отражать длительность существования L1-состояния, а не количество пришедших тиков.

Флор не считается существующим автоматически. После начала аномалии его оценка замораживается. При смене уровня, недостаточном тихом покрытии или загрязнении окна флор признаётся недоступным.

Условие

$$
Q_{50}^{\,w}(s(t)-F_t)\approx 0
$$

означает медианное центрирование тихого распределения. Оно не гарантирует, что

$$
\mathbb E[s(t)-F_t]=0,
$$

и не гарантирует центрирование распределения в специально выбранный момент выхода.

### P4. Исполнение моделируется отдельно на входе и выходе

Сделка содержит два отдельных перехода от сигнала к исполнению:

$$
s(t_{\mathrm{in}})
\longrightarrow
s_{\mathrm{in}}^{\mathrm{fill}},
$$

$$
s(t_{\mathrm{out}})
\longrightarrow
s_{\mathrm{out}}^{\mathrm{fill}}.
$$

Фактически полученные спреды определяются моделью исполнения:

$$
s_{\mathrm{in}}^{\mathrm{fill}}
=
s^{\mathrm{fill}}
\left(
t_{\mathrm{in}}+T_{\mathrm{in}}
\right),
$$

$$
s_{\mathrm{out}}^{\mathrm{fill}}
=
s^{\mathrm{fill}}
\left(
t_{\mathrm{out}}+T_{\mathrm{out}}
\right).
$$

В историческом симуляторе $s^{\mathrm{fill}}(t+T)$ означает исполнение по первому допустимому L1-состоянию с временной меткой не раньше $t+T$.

Ошибки исполнения входа и выхода определяются отдельно:

$$
\varepsilon_{\mathrm{in}}
=
s_{\mathrm{in}}^{\mathrm{fill}}
-
s(t_{\mathrm{in}}),
$$

$$
\varepsilon_{\mathrm{out}}
=
s_{\mathrm{out}}^{\mathrm{fill}}
-
s(t_{\mathrm{out}}).
$$

Они могут иметь разные распределения и по-разному зависеть от величины спреда, режима рынка и доступной ликвидности. Поэтому их нельзя заменять одним общим `latency haircut`.

Для этапа $k\in{\mathrm{in},\mathrm{out}}$ определим коэффициент участия заявки в согласованной глубине двух ног:

$$
\rho_k=
\frac{q}{V_k^{\mathrm{matched}}}.
$$

При условии

$$
\rho_k \ll 1
$$

исполнение можно приближённо описывать через наблюдаемый L1, задержку и статистически измеренный пессимисистичный квантиль.

Требуемую позицию $M$ в дальнейшем можно делить на части:

$$
q_j=
\min\left(
\frac{M}{N},
\eta V_{t_j}^{\mathrm{matched}},
M_{\mathrm{remaining}}
\right).
$$

Увеличение $N$ уменьшает участие отдельной заявки в доступной ликвидности, но не устраняет:

* комиссии;
* изменение спреда за время задержки;
* исчезновение доступной глубины;
* риск несинхронного исполнения ног;
* изменение режима между частями позиции;
* риск недостаточной ликвидности при выходе.

Перед исполнением каждой следующей части должны заново оцениваться состояние рынка, доступная глубина и оставшийся край.

### P5. Тики не являются независимыми наблюдениями

Последовательные L1-состояния, сигналы одной монеты и события одного рыночного шока статистически зависимы.

Поэтому эффективный размер выборки определяется не количеством строк, а количеством независимых:

* временных блоков;
* монет;
* эпизодов;
* shock-кластеров.

Неопределённость должна оцениваться кластерным или блочным ресэмплингом. Устойчивость результата проверяется как минимум по последовательному исключению отдельных дней, монет и эпизодов.

Эффект, который существует только в объединённой выборке и исчезает при таком исключении, не считается переносимым свойством стратегии.

### P6. Отсутствующие или невалидные данные означают неопределённость

Отсутствие тиков не интерпретируется как нулевое изменение или спокойный рынок:

$$
\text{нет наблюдения}
\ne
\Delta s=0.
$$

Спред и результат сделки считаются наблюдаемыми только при выполнении контракта качества данных:

* доступны обе ноги;
* соблюдены ограничения возраста ног;
* соблюдено ограничение временного перекоса;
* измерение не пересекает неразмеченный разрыв;
* направление и единицы измерения спреда определены однозначно.

Невалидное наблюдение исключается или явно цензурируется. Оно не может автоматически считаться ни успешным возвратом, ни отсутствием движения.

Если полный результат сделки нельзя восстановить, такой исход не должен бесследно удаляться из выборки.

### P7. Результат сделки зависит от времён сигналов и задержек исполнения

Время $t_{\mathrm{out}}$ является временем срабатывания принятой политики выхода. Оно может определяться достижением флора, сменой режима, риск-условием или максимальным временем удержания.

Если потребуется максимальное время удержания, для него используется отдельное обозначение $H_{\max}$. Символ $T$ сохраняется за задержкой исполнения.

Фактически захваченный спред определяется как

$$
G
\left(
t_{\mathrm{in}},
t_{\mathrm{out}},
T_{\mathrm{in}},
T_{\mathrm{out}}
\right)
=
s^{\mathrm{fill}}
\left(
t_{\mathrm{in}}+T_{\mathrm{in}}
\right)
-
s^{\mathrm{fill}}
\left(
t_{\mathrm{out}}+T_{\mathrm{out}}
\right).
$$

Через ошибки исполнения его можно записать как

$$
G=
\left[
s(t_{\mathrm{in}})
-
s(t_{\mathrm{out}})
\right]
+
\varepsilon_{\mathrm{in}}
-
\varepsilon_{\mathrm{out}}.
$$

Если выход задаётся относительно флора, определим отклонение сигнала выхода от целевого уровня:

$$
d_{\mathrm{exit}}
=
s(t_{\mathrm{out}})
-
F_{\mathrm{exit}}.
$$

Тогда

$$
G=
\left[
s(t_{\mathrm{in}})
-
F_{\mathrm{exit}}
\right]
+
\varepsilon_{\mathrm{in}}
-
d_{\mathrm{exit}}
-
\varepsilon_{\mathrm{out}}.
$$

Здесь:

* $s(t_{\mathrm{in}})-F_{\mathrm{exit}}$ — виртуальный край, наблюдаемый в момент входа;
* $\varepsilon_{\mathrm{in}}$ — изменение края между сигналом и исполнением входа;
* $d_{\mathrm{exit}}$ — отклонение сигнала выхода от целевого флора;
* $\varepsilon_{\mathrm{out}}$ — изменение спреда между сигналом и исполнением выхода.

В детерминированной модели результат сделки записывается как

$$
\Pi(t_{\mathrm{in}},t_{\mathrm{out}},T,q).
$$

В стохастической модели задержки записываются явно:

$$
\Pi
\left(
t_{\mathrm{in}},
t_{\mathrm{out}},
T_{\mathrm{in}},
T_{\mathrm{out}},
q
\right).
$$

Если спред выражен в долях номинала, чистый результат равен

$$
\Pi
\left(
t_{\mathrm{in}},
t_{\mathrm{out}},
T_{\mathrm{in}},
T_{\mathrm{out}},
q
\right)
=
qG
-
C_{\mathrm{fee}}^{(4)}(q)
-
C_{\mathrm{residual}}
\left(
t_{\mathrm{in}},
t_{\mathrm{out}},
T_{\mathrm{in}},
T_{\mathrm{out}},
q
\right).
$$

Здесь $C_{\mathrm{fee}}^{(4)}$ включает четыре тейкер-комиссии, а $C_{\mathrm{residual}}$ содержит только те издержки, которые ещё не представлены в модели заполнения.

Если $s_{\mathrm{in}}^{\mathrm{fill}}$ и $s_{\mathrm{out}}^{\mathrm{fill}}$ уже рассчитаны по фактическим ценам исполнения, повторно добавлять учтённое проскальзывание в $C_{\mathrm{residual}}$ нельзя.

Даже если отклонения около флора центрированы в безусловной тихой выборке, из этого не следует

$$
Q_{50}
\left(
\varepsilon_{\mathrm{out}}
\mid
t_{\mathrm{out}}
\text{ является первым сигналом выхода}
\right)
=0.
$$

Момент $t_{\mathrm{out}}$ выбирается политикой выхода, поэтому распределение ошибки выхода может отличаться от распределения ошибки в случайной точке тихого режима.

Величины $\varepsilon_{\mathrm{in}}$, $d_{\mathrm{exit}}$ и $\varepsilon_{\mathrm{out}}$ должны оцениваться совместно на одной траектории. Подмена их независимыми средними значениями может занизить хвостовой риск.

В момент $t_{\mathrm{in}}$ будущее значение $t_{\mathrm{out}}$ ещё неизвестно. Поэтому вход определяется по условному распределению полного результата при фиксированной политике выхода:

$$
E_{\alpha,\beta}(t_{\mathrm{in}},q)
=
\operatorname{LCB}_{\beta}
\left[
Q_{\alpha}
\left(
\Pi
\left(
t_{\mathrm{in}},
t_{\mathrm{out}},
T_{\mathrm{in}},
T_{\mathrm{out}},
q
\right)
\mid
X_{t_{\mathrm{in}}}
\right)
\right].
$$

Вход допустим только если

$$
E_{\alpha,\beta}(t_{\mathrm{in}},q)>0,
$$

а оценки режима, флора, качества данных и исполнения признаны валидными.

Параметры $q$, $\alpha$, $\beta$, ширина тихого окна, политика выхода и закон задержки $\mathcal D_T$ являются явными степенями свободы модели. Их значения должны фиксироваться до проверки на устойчивость.

### Прикладной критерий состоятельности модели

Модель считается полезной не тогда, когда она хорошо объясняет исторический график, а когда она:

1. причинно вычисляется в момент принятия решения;
2. сохраняет калибровку на независимых эпизодах;
3. моделирует исполнение отдельно на входе и выходе;
4. учитывает зависимость результата от $t_{\mathrm{in}}$, $t_{\mathrm{out}}$, $T_{\mathrm{in}}$ и $T_{\mathrm{out}}$;
5. учитывает комиссии и остаточные издержки без двойного счёта;
6. различает статистическое отклонение и экономически исполнимый вход;
7. умеет возвращать решение «данных недостаточно — сделку не совершать».



## План построения моделей результата сделки $\Pi$ и статистического края $\mathcal E$

### Цель

Необходимо построить прикладную модель, которая в момент сигнала $t_{\mathrm{in}}$:

1. определяет текущее состояние и режим рынка;
2. выбирает допустимую для этого режима торговую политику;
3. оценивает распределение полного результата сделки;
4. разрешает вход только при положительной консервативной оценке края.

Будем различать:

* $\Pi$ — случайный результат будущей сделки;
* $\pi_i$ — конкретный исторический или симулированный результат сделки $i$;
* $\mathcal E$ — консервативная нижняя оценка края.

### Общая последовательность

Построение модели разбивается на этапы:

1. контракт качества данных;
2. причинное описание состояния $X_t$;
3. определение и проверка рыночных режимов;
4. построение индивидуальной политики для каждого режима;
5. симуляция реализаций $\pi_i$;
6. оценка условного распределения $\Pi$;
7. расчёт нижней доверительной границы $\mathcal E$;
8. проверка всей конструкции на устойчивость.

---

## Этап 1. Контракт наблюдаемого состояния

Любая классификация режима должна вычисляться только по информации, доступной к моменту $t$:

$$
X_t=\Phi(\mathcal O_{\le t}).
$$

В исходное состояние могут входить:

* текущий направленный спред $s(t)$;
* оценка флора $\widehat F_t$;
* расстояние до флора;
* робастный масштаб $\widehat\sigma_t$;
* нормированное отклонение $z_t$;
* персистентность отклонения;
* скорость и направление движения флора;
* L1-номинал обеих ног;
* возраст и перекос котировок;
* частота обновления L1;
* ширина общерыночного движения;
* признаки разрыва или загрязнения данных.

Невалидное состояние должно образовывать отдельный класс:

$$
C_t=\text{invalid},
$$

а не смешиваться с тихим рынком.

---

## Этап 2. Разделение онлайн-режима и предыдыдущего класса эпизода

Необходимо разделить два разных объекта.

### Онлайн-режим

Онлайн-режим определяется в момент принятия решения:

$$
C_t=c(X_t).
$$

Он может принимать, например, значения:

* `quiet_stable`;
* `dislocation_candidate`;
* `level_shift_suspected`;
* `common_shock`;
* `invalid_or_unknown`.

Точное число и содержание классов ещё не фиксируются. Оно должно определяться отдельным экспериментом.

### Предыдущий класс

После завершения эпизода можно определить его фактический исход:

* возврат к прежнему флору;
* переход к новому уровню;
* общий рыночный шок;
* цензурированный эпизод;
* артефакт данных.

Предыдущий класс нужен для исследования и проверки онлайн-классификации, но не может использоваться как признак входа.

---

## Этап 3. Решение задачи кластеризации режимов

Кластеризация должна отвечать не на вопрос «какие графики визуально похожи», а на вопрос:

> В каких состояниях будущая динамика спреда подчиняется приблизительно одному условному закону?

Для режима $c$ рассматривается нормированная траектория:

$$
Z_c(u)=
\frac{s(t+\tau_c u)-F_c}{\sigma_c}.
$$

Объединение состояний в один кластер допустимо, если после нормировки сохраняются приблизительно одинаковые:

* распределения изменений спреда;
* вероятности движения к флору и от флора;
* выжившие-функции времени возврата;
* вероятность смены уровня;
* распределения ошибок исполнения;
* частота цензурированных исходов.

Первую версию кластеризации следует строить по небольшому числу робастных причинных признаков, без нейросети:

$$
X_t^{\mathrm{regime}}
=
\left(
z_t,\,
\frac{d\widehat F_t}{dt},\,
\text{persistence}_t,\,
\text{market breadth}_t,\,
\text{depth}_t,\,
\text{data quality}_t
\right).
$$

Кластеризация может быть:

* правиловой;
* основанной на робастных расстояниях;
* алгоритмической, если правилового разделения недостаточно.

Число кластеров выбирается не по PnL и не только по абсолютному скору. Следует выбирать минимальное разделение, которое:

1. стабильно при исключении отдельных монет и эпизодов;
2. уменьшает внутрикластерную неоднородность будущей динамики;
3. проверяемо на устойчивость;
4. не создаёт классы, состоящие почти полностью из одного эпизода.

Если два кластера после нормировки имеют близкие законы, их следует объединить. Если один кластер содержит несовместимые распределения, его следует разделить или признать состояние неопределённым.

---

## Этап 4. Индивидуальная спецификация режима

После фиксации режима $c$ для него определяется отдельный набор объектов:

$$
\mathcal M_c=
\left(
F_c,\,
P_c^{\mathrm{exit}},\,
\mathcal D_{T,c},\,
\mathcal D_{\mathrm{fill},c},\,
H_{\max,c},\,
q_c
\right).
$$

Здесь:

* $F_c$ — способ оценки флора;
* $P_c^{\mathrm{exit}}$ — политика выхода;
* $\mathcal D_{T,c}$ — модель задержки;
* $\mathcal D_{\mathrm{fill},c}$ — модель исполнения;
* $H_{\max,c}$ — максимальное время удержания;
* $q_c$ — допустимый размер позиции.

Не каждый режим обязан иметь торговую политику. Допустим результат:

```text
level_shift_suspected → no trade
common_shock          → no trade
invalid_or_unknown    → no trade
```

Для `dislocation_candidate` необходимо отдельно проверить:

* существует ли доступный флор;
* стабилен ли он;
* положительна ли вероятность возврата;
* достаточно ли времени живёт исполнимый спред;
* остаётся ли край после входного и выходного исполнения.

---

## Этап 5. Формирование исторических реализаций $\pi_i$

Для каждого допустимого исторического сигнала $i$ симулируется полный жизненный цикл сделки.

Вход:

$$
s_{\mathrm{in},i}^{\mathrm{fill}}
=
s^{\mathrm{fill}}
\left(
t_{\mathrm{in},i}+T_{\mathrm{in},i}
\right).
$$

Выход:

$$
s_{\mathrm{out},i}^{\mathrm{fill}}
=
s^{\mathrm{fill}}
\left(
t_{\mathrm{out},i}+T_{\mathrm{out},i}
\right).
$$

Захваченный спред:

$$
G_i=
s_{\mathrm{in},i}^{\mathrm{fill}}
-
s_{\mathrm{out},i}^{\mathrm{fill}}.
$$

Реализация результата:

$$
\pi_i
=
q_iG_i
-
C_{\mathrm{fee},i}^{(4)}
-
C_{\mathrm{residual},i}.
$$

Каждая строка результата должна содержать:

```text
trade_id
episode_id
shock_cluster_id
regime_id
base_coin
direction
t_in
t_out
T_in
T_out
s_signal_in
s_fill_in
F_exit
s_signal_out
s_fill_out
q
fees
residual_cost
pi
exit_reason
data_valid
censored
```

Недостижение флора не удаляется из выборки. Оно получает заранее определённый `exit_reason` и результат в соответствии с политикой $H_{\max}$.

---

## Этап 6. Оценка условного распределения $\Pi$

Для режима $c$ оценивается:

$$
\mathcal L
\left(
\Pi
\mid
C_t=c,\,
X_t^{\mathrm{trade}},\,
q
\right),
$$

где $X_t^{\mathrm{trade}}$ содержит признаки, необходимые уже внутри режима:

* оставшийся край до флора;
* персистентность;
* L1-номинал;
* ожидаемое время возврата;
* устойчивость флора;
* параметры исполнения.

Первая версия может использовать фиксированные страты:

```text
regime_id
× edge_bin
× persistence_bin
× depth_bin
× direction
```

Границы страт фиксируются до просмотра результатов. Если в страте недостаточно независимых эпизодов, распределение считается недоступным:

```text
insufficient support → no trade
```

---

## Этап 7. Расчёт консервативного края

Для текущего состояния оценивается нижний квантиль результата:

$$
\widehat\theta_\alpha(X_t,q)
=
\widehat Q_\alpha
\left(
\Pi\mid C_t,X_t,q
\right).
$$

Затем кластерным bootstrap оценивается его нижняя доверительная граница:

$$
\mathcal E_{\alpha,\beta}(t_{\mathrm{in}},q)
=
\operatorname{LCB}_\beta
\left[
\widehat\theta_\alpha(X_{t_{\mathrm{in}}},q)
\right].
$$

Если $\beta$ обозначает доверительный уровень, то:

$$
\operatorname{LCB}_\beta
=
Q_{1-\beta}
\left(
\widehat\theta_\alpha^{*}
\right).
$$

Bootstrap выполняется по независимым эпизодам или шок-кластерам, а не по отдельным сделкам и тикам.

Итоговый вход разрешается только при одновременном выполнении:

$$
\begin{aligned}
&\text{data valid},\\
&\text{regime supported},\\
&\text{floor available},\\
&\text{execution supported},\\
&\mathcal E_{\alpha,\beta}(t_{\mathrm{in}},q)>0.
\end{aligned}
$$

---

## Этап 8. Holdout и проверка переносимости

Разделение должно выполняться по независимым эпизодам, а не по строкам:

* train — определение режимов и оценка моделей;
* calibration — фиксация степеней свободы;
* test — финальная проверка без изменений методологии.

На holdout проверяются:

* стабильность назначения режимов;
* доля состояний `unknown`;
* калибровка условных квантилей $\Pi$;
* покрытие LCB;
* устойчивость по монетам и дням;
* концентрация результата в отдельных эпизодах;
* частота недостижения выхода;
* распределение времени удержания;
* ошибки исполнения входа и выхода.

PnL holdout является результатом проверки, но не критерием выбора кластеризации.

---

## Реестр степеней свободы

До финального теста должны быть зафиксированы:

```text
regime_features
regime_partition_rule
number_of_regimes
floor_estimator
floor_window
exit_policy
H_max
fill_contract
latency_law
position_size
conditional_strata
alpha
beta
bootstrap_unit
minimum_support
holdout_split
```

Изменение любого из этих параметров после просмотра test переводит результат обратно в исследовательский статус.

---

## Первый измеримый шаг

Сначала необходимо построить таблицу состояний с одной строкой на независимый anchor или начало эпизода:

```text
anchor_id
episode_id
shock_cluster_id
base_coin
direction
ts_utc
data_valid
floor_available
floor_level
floor_scale
z
floor_slope
persistence
market_breadth
l1_notional
quote_age
ex_post_episode_class
```

После этого первый эксперимент должен проверить:

> Существует ли минимальное устойчивое разделение состояний, внутри которого нормированные распределения движения к флору, времени возврата и смены уровня воспроизводятся при leave-one-episode и leave-one-coin проверках?

До прохождения этого гейта строить индивидуальные модели $\Pi$ и положительный торговый $\mathcal E$ преждевременно.


## Задача 1. Автоматическая сегментация и первичная классификация эпизодов спреда

### 1. Постановка задачи

Необходимо построить детектор, который без предварительного ручного выбора эпизодов преобразует непрерывный ряд направленного спреда в каталог:

$$
\mathcal D_\theta:
s_d(t)
\longrightarrow
\left\{
E_1,E_2,\ldots,E_N
\right\},
$$

где:

* $d\in{\mathrm{long},\mathrm{short}}$ — направление;
* $E_i$ — автоматически найденный эпизод;
* $\theta$ — набор параметров сегментации и классификации.

Для каждого набора параметров $\theta$ детектор должен:

1. найти границы всех эпизодов на заданном непрерывном датасете;
2. отделить устойчивые отклонения от коротких прострелов;
3. определить факт возврата к прежнему флору;
4. проверить возможность формирования нового флора;
5. присвоить эпизоду первичный морфологический класс;
6. сформировать визуальный атлас и статистический отчёт.

Ручной выбор отдельных эпизодов до запуска детектора не используется.

---

### 2. Исследовательская гипотеза

Предполагается, что существует не одна точная комбинация порогов, а связная область параметров

$$
\Theta_*
\subset
\Theta,
$$

внутри которой:

* находятся приблизительно одни и те же крупные эпизоды;
* границы эпизодов изменяются плавно;
* короткие прострелы не превращаются в самостоятельные эпизоды;
* первичные классы сохраняют морфологический смысл;
* статистические свойства классов воспроизводятся по монетам и временным блокам.

Если небольшое изменение параметров полностью перестраивает каталог, выбранная система метрик не считается надёжной.

---

### 3. Принцип выбора параметров

На первом исследовательском этапе визуальная оценка допустима как источник экспертной информации.

Однако объектом визуального выбора должен быть не отдельный понравившийся эпизод, а поведение всей системы при изменении параметров:

$$
\theta
\longrightarrow
\text{каталог эпизодов}
\longrightarrow
\text{морфология классов}
\longrightarrow
\text{устойчивость каталога}.
$$

Параметры выбираются по визуальной и статистической устойчивости разделения, а не по PnL найденных эпизодов.

На этом этапе PnL и будущая доходность эпизодов не отображаются в отчёте.

---

## 4. Предэпизодный флор

Для каждой монеты и направления по причинному тихому окну оцениваются:

$$
F_{0,\mathrm{mid}}
=
Q_{50}^{\,w}(s_d),
$$

$$
\mathcal F_0
=
\left[
Q_{25}^{\,w}(s_d),
Q_{75}^{\,w}(s_d)
\right],
$$

$$
\sigma_0
=
1.4826\operatorname{wMAD}(s_d).
$$

Веса отражают время существования L1-состояния, а не количество тиков.

После подтверждения начала эпизода значения

$$
F_{0,\mathrm{mid}},
\qquad
\mathcal F_0,
\qquad
\sigma_0
$$

замораживаются до завершения классификации эпизода.

Ширина максимального предэпизодного окна $H_{\mathrm{floor}}$ является параметром эксперимента. Флор и остальные параметры предэпизодного окна будут рассчитываться по этой ширине: $H_{\mathrm{floor}}$ если ширина тихого эпизода больше $H_{\mathrm{floor}}$, в ином случае берется полная ширина тихого окна. Предварительный центральный кандидат:

$$
H_{\mathrm{floor}}=12\ \text{часов}.
$$

Для проверки чувствительности рассматриваются соседние масштабы, например $6$, $12$ и $24$ часа.

---

## 5. Нормированное отклонение от флора

Для положительного направленного спреда определяется расстояние от верхней границы тихого диапазона:

$$
z_+(t)
=
\frac{
\left[
s_d(t)-F_{0,75}
\right]_+
}{
\sigma_0
},
$$

где

$$
[x]_+=\max(0,x).
$$

Величина $z_+(t)$ измеряет амплитуду отклонения, но не его продолжительность. Поэтому одного порога по $z_+$ недостаточно.

---

## 6. Интегральные метрики устойчивого отклонения

### 6.1 Доля времени выше базового уровня

$$
O_W(t)
=
\frac{1}{W}
\int_{t-W}^{t}
\mathbf 1
\left[
z_+(u)\ge z_{\mathrm{base}}
\right]du.
$$

$O_W(t)$ показывает, какую долю окна $W$ спред действительно находился далеко от флора.

Короткий прострел длительностью $\delta\ll W$ даст малое значение порядка

$$
O_W\sim\frac{\delta}{W},
$$

даже если его амплитуда велика.

### 6.2 Интегральная площадь отклонения

$$
I_{W,1}(t)
=
\frac{1}{W}
\int_{t-W}^{t}
\left[
z_+(u)-z_{\mathrm{base}}
\right]_+du.
$$

Эта метрика одновременно учитывает амплитуду и продолжительность отклонения.

### 6.3 Усиленная метрика больших отклонений

Опционально:

$$
I_{W,2}(t)
=
\frac{1}{W}
\int_{t-W}^{t}
\left[
z_+(u)-z_{\mathrm{base}}
\right]_+^2du.
$$

$I_{W,2}$ сильнее реагирует на высокие значения $z_+$ и может понадобиться, если линейная площадь плохо разделяет умеренные длительные отклонения и короткие экстремальные скачки.

В первой версии основной интегральной метрикой следует использовать $I_{W,1}$. Квадратичную версию нужно рассматривать как отдельный arm, а не добавлять автоматически.

---

## 7. Определение начала эпизода

Момент подтверждения эпизода определяется как

$$
t_{\mathrm{confirmed}}
=
\inf
\left\{
t:
\begin{array}{l}
z_+(t)\ge z_{\mathrm{base}},\\
O_W(t)\ge O_{\min},\\
I_{W,1}(t)\ge I_{\min}
\end{array}
\right\}.
$$

Условия имеют разный смысл:

* $z_{\mathrm{base}}$ — отклонение достигло существенной амплитуды;
* $O_{\min}$ — отклонение занимало существенную часть окна;
* $I_{\min}$ — накопленная амплитуда отклонения достаточно велика.

После подтверждения граница начала эпизода восстанавливается внутри предшествующего окна:

$$
t_{\mathrm{onset}}
=
\inf
\left\{
u\in[t_{\mathrm{confirmed}}-W,t_{\mathrm{confirmed}}]:
z_+(u)\ge z_{\mathrm{base}}
\right\}.
$$

При этом сохраняются оба времени:

```text
onset_ts
confirmed_ts
detection_delay
```

где

$$
D_{\mathrm{detect}}
=
t_{\mathrm{confirmed}}-t_{\mathrm{onset}}.
$$

Online-стратегия сможет действовать только в $t_{\mathrm{confirmed}}$, тогда как $t_{\mathrm{onset}}$ используется для ex-post анализа границ эпизода.

---

## 8. Определение возврата

Возврат не фиксируется по одному пересечению флора.

Вводится доля времени внутри замороженного тихого диапазона:

$$
R_{W_{\mathrm{return}}}(t)
=
\frac{1}{W_{\mathrm{return}}}
\int_{t-W_{\mathrm{return}}}^{t}
\mathbf 1
\left[
s_d(u)\in\mathcal F_0
\right]du.
$$

Подтверждённый возврат:

$$
t_{\mathrm{return}}
=
\inf
\left\{
t>t_{\mathrm{onset}}:
R_{W_{\mathrm{return}}}(t)
\ge R_{\min}
\right\}.
$$

Повторные пики до подтверждённого возврата относятся к одному эпизоду.

Если после краткого возврата новый выход происходит раньше периода повторного спокойствия $G_{\mathrm{merge}}$, интервалы объединяются в один эпизод.

---

## 9. Первичная классификация

### `transient_dislocation`

Эпизод устойчиво ушёл от старого флора и подтвердил возврат:

$$
t_{\mathrm{return}}-t_{\mathrm{onset}}
\le H_{\mathrm{return}}.
$$

### `persistent_shift`

Эпизод не вернулся к старому флору за $H_{\mathrm{return}}$, после чего сформировался новый устойчивый флор $\mathcal F_1$.

Нормированное смещение:

$$
\Delta_F
=
\frac{
\left|
F_{1,\mathrm{mid}}-F_{0,\mathrm{mid}}
\right|
}{
\sigma_0
}.
$$

Для подтверждения требуется:

$$
\Delta_F\ge\Delta_{F,\min},
$$

$$
\left|
\frac{dF_1}{dt}
\right|
\le D_{F,\max}.
$$

### `unresolved_nonreturn`

Возврата к старому флору нет, но:

* новый флор не сформировался;
* новый уровень продолжает двигаться;
* недостаточно постэпизодного покрытия;
* результат нельзя однозначно классифицировать.

### `quiet_stable`

Тихими считаются валидные интервалы вне найденных эпизодов, для которых:

$$
O_W(t)<O_{\min},
$$

$$
I_{W,1}(t)<I_{\min},
$$

а флор остаётся локально стабильным.

Короткие высокие прострелы могут присутствовать в `quiet_stable`, если они не создают достаточной временной площади отклонения.

---

## 10. Параметры системы

Параметры детектора:

$$
\theta_{\mathrm{detect}}
=
\left(
H_{\mathrm{floor}},
z_{\mathrm{base}},
W,
O_{\min},
I_{\min}
\right).
$$

Параметры границ:

$$
\theta_{\mathrm{boundary}}
=
\left(
W_{\mathrm{return}},
R_{\min},
G_{\mathrm{merge}}
\right).
$$

Параметры классификации:

$$
\theta_{\mathrm{class}}
=
\left(
H_{\mathrm{return}},
H_{\mathrm{new\ floor}},
\Delta_{F,\min},
D_{F,\max}
\right).
$$

Полный вектор:

$$
\theta
=
\left(
\theta_{\mathrm{detect}},
\theta_{\mathrm{boundary}},
\theta_{\mathrm{class}}
\right).
$$

Все параметры являются степенями свободы и должны сохраняться вместе с результатом каждого прогона.

---

## 11. Порядок sweep-поиска

Не следует сразу запускать полный декартов перебор всех параметров.

### Sweep A. Детекция

Сначала изменяются:

$$
H_{\mathrm{floor}},
\quad
z_{\mathrm{base}},
\quad
W,
\quad
O_{\min},
\quad
I_{\min}.
$$

Остальные параметры фиксируются в широких технических значениях.

Цель — отделить короткие прострелы от устойчивых отклонений.

### Sweep B. Границы

После выбора устойчивой области детектора изменяются:

$$
W_{\mathrm{return}},
\quad
R_{\min},
\quad
G_{\mathrm{merge}}.
$$

Цель — избежать дробления одного эпизода и ложных возвратов по одному тику.

### Sweep C. Классы

После фиксации границ изменяются:

$$
H_{\mathrm{return}},
\quad
H_{\mathrm{new\ floor}},
\quad
\Delta_{F,\min},
\quad
D_{F,\max}.
$$

Цель — разделить возвратные дислокации, подтверждённые смены уровня и неразрешённые исходы.

---

## 12. Визуальный протокол

Для каждого $\theta$ автоматически строится полный визуальный отчёт.

### Атлас эпизодов

Для каждого эпизода отображаются:

* сырой направленный спред;
* замороженный старый флор $\mathcal F_0$;
* потенциальный новый флор $\mathcal F_1$;
* $t_{\mathrm{onset}}$;
* $t_{\mathrm{confirmed}}$;
* $t_{\mathrm{return}}$;
* значения $z_+(t)$;
* интегральная метрика $I_{W,1}(t)$;
* присвоенный класс.

Эпизоды выбираются для показа не по PnL, а следующим образом:

* все найденные эпизоды, если их число обозримо;
* иначе детерминированная случайная выборка;
* центральные представители класса;
* крайние значения по длительности и площади;
* случаи, меняющие класс при соседних параметрах.

### Фазовая карта параметров

Для каждой комбинации параметров отображаются:

* число эпизодов;
* суммарное время внутри эпизодов;
* доля `quiet_stable`;
* доля `transient_dislocation`;
* доля `persistent_shift`;
* доля `unresolved_nonreturn`;
* медианная длительность;
* fragmentation rate;
* число эпизодов, созданных одной монетой;
* число эпизодов, исчезающих при малом изменении параметров.

### Карта переходов

Для соседних наборов параметров строится таблица переходов:

```text
transient → transient
transient → quiet
transient → unresolved
unresolved → persistent
one episode → several fragments
several fragments → one episode
```

Это позволяет видеть не только итоговые пропорции классов, но и конкретные причины перестройки каталога.

---

## 13. Критерии визуальной адекватности

До просмотра результатов фиксируется следующая рубрика.

Разделение считается визуально осмысленным, если:

1. короткие одиночные прострелы преимущественно остаются вне эпизодов;
2. длительный уход спреда образует один связный эпизод;
3. повторные пики до устойчивого возврата не дробятся;
4. один тик внутри флора не завершает эпизод;
5. отсутствие возврата автоматически не объявляется новым флором;
6. подтверждённая смена уровня визуально соответствует новому устойчивому диапазону;
7. один класс не формируется почти исключительно одной монетой;
8. небольшое изменение параметров не перестраивает весь каталог;
9. выбранное разделение сохраняет смысл на случайно выбранных, а не только крайних примерах.

«Визуальное удовлетворение» в отчёте должно означать выполнение этой рубрики, а не субъективный выбор наиболее красивого набора графиков.

---

## 14. Статистическое сопровождение визуального выбора

Статистика на этом этапе не создаёт классы, а проверяет устойчивость экспертного разделения.

Для каждого класса выводятся распределения:

$$
z_{\max},
\quad
I_{\mathrm{episode}},
\quad
D_{\mathrm{episode}},
\quad
T_{\mathrm{return}},
\quad
\Delta_F,
\quad
D_{\mathrm{detect}}.
$$

Для соседних наборов параметров оцениваются:

* доля совпавших эпизодов;
* временной Jaccard overlap;
* разброс $t_{\mathrm{onset}}$ и $t_{\mathrm{return}}$;
* доля эпизодов, сменивших класс;
* устойчивость по монетам и UTC-дням;
* концентрация класса в крупнейшем эпизоде;
* различие survival-кривых возврата.

Искомым результатом является не точка максимума некоторого score, а область, где статистические характеристики и визуальная морфология меняются плавно.

---

## 15. Защита от визуальной подгонки

Работа разделяется на два окна.

### Discovery

На discovery-части разрешены:

* просмотр всех визуальных отчётов;
* изменение набора метрик;
* изменение диапазонов параметров;
* формулировка визуальной рубрики;
* выбор устойчивой области $\Theta_*$.

### Holdout

После discovery фиксируются:

* формулы метрик;
* диапазон $\Theta_*$;
* центральный набор $\theta_*$;
* правила границ;
* правила классификации;
* визуальная рубрика.

На holdout выполняется один автоматический прогон без изменения методологии.

Если визуальная структура и распределения классов не воспроизводятся, задача сегментации считается не решённой. Параметры возвращаются в исследовательский статус.

---

## 16. Выбор итоговых параметров

Из устойчивой области $\Theta_*$ выбирается не самый прибыльный и не самый визуально эффектный набор, а центральная конфигурация:

$$
\theta_*
=
\operatorname{center}(\Theta_*).
$$

Она должна иметь запас до границ области устойчивости.

Если удовлетворительный результат существует только в одной узкой точке параметров, такой детектор считается переобученным:

$$
\Theta_*\approx\{\theta_*\}
\quad\Longrightarrow\quad
\text{detector rejected}.
$$

---

## 17. Выходные артефакты

Эксперимент должен сформировать:

```text
episode_catalog
threshold_sweep_summary
episode_transition_table
cluster_distribution_table
episode_atlas.html
parameter_phase_map.html
holdout_report.md
```

Минимальные поля каталога:

```text
episode_id
base_coin
direction
onset_ts
confirmed_ts
peak_ts
return_ts
end_ts
episode_class
floor_mid
floor_q25
floor_q75
floor_scale
z_max
excursion_area
duration
return_time
new_floor_mid
floor_shift
parameter_set_id
data_valid
right_censored
manual_review_required
```

Дыры не образуют отдельный рыночный класс. Они сохраняются как признаки качества и цензурирования результата.

---

## 18. Критерий завершения задачи

Задача считается успешно решённой, если:

1. весь заданный ряд обрабатывается без ручного выбора эпизодов;
2. существует связная область устойчивых параметров $\Theta_*$;
3. визуальная морфология соответствует заранее зафиксированной рубрике;
4. соседние параметры дают близкие каталоги;
5. классы имеют воспроизводимые статистические распределения;
6. результат сохраняется на holdout;
7. PnL не использовался для выбора метрик и порогов.

Если устойчивой области нет, это означает не необходимость точнее подкрутить один порог, а недостаточность выбранного набора метрик или неверное определение флора.

---

## Первый конкретный эксперимент

Первая проверяемая гипотеза:

> Интегральная time-weighted метрика отделяет короткие высокие прострелы от устойчивых уходов спреда лучше и устойчивее, чем один амплитудный порог $z_{\mathrm{base}}$.

Сравниваются три arms:

* A: только $z_{\mathrm{base}}$;
* B: $z_{\mathrm{base}}+O_W$;
* C: $z_{\mathrm{base}}+O_W+I_{W,1}$.

Сравнение выполняется по:

* фазовым картам параметров;
* визуальной рубрике;
* fragmentation rate;
* устойчивости границ;
* устойчивости классов;
* воспроизводимости на holdout.

Первый результат должен определить, даёт ли интегральная метрика устойчивую топологию эпизодов. Только после этого фиксируются окончательные правила сегментации и начинается статистический анализ полученных классов.


# Эксперимент 2.2-Z1: переносимость нормировки тихого спреда

## 1. Статус и область эксперимента

Эксперимент относится к исследовательской ветке Track M, gear 2.2.

Цель — проверить статистическую переносимость нормированной метрики спреда между тихими режимами:

* одной монеты;
* разных монет одного класса ликвидности;
* всего рынка, включая BTC, SOL и альткоины.

Эксперимент не проверяет PnL, торговый вход, параметры `VARIATION`, $W$, $O_{\min}$, $I_{\min}$ и не меняет канонический симулятор.

Результатом является решение о том, какую следующую гипотезу имеет смысл исследовать:

1. один глобальный закон нормированного тихого спреда;
2. отдельные законы по классам ликвидности;
3. автоматическая per-coin процентильная нормировка;
4. отказ от текущей median/MAD-нормировки.

---

## 2. Исследуемая гипотеза

Пусть для монеты $c$, направления $d$ и тихого интервала $e$ наблюдается исполнимый L1-спред

$$
s_{c,d,e}(t).
$$

Локальный уровень и масштаб тихого режима:

$$
F_{c,d,e}
=
Q_{0.50}^{w}\left(s_{c,d,e}\right),
$$

$$
\sigma_{c,d,e}
=
1.4826\,
\operatorname{wMAD}\left(s_{c,d,e}\right).
$$

Здесь все квантили взвешиваются по времени существования L1-состояния, а не по количеству тиков.

Центрированный нормированный спред:

$$
r_{c,d,e}(t)
=
\frac{
s_{c,d,e}(t)-F_{c,d,e}
}{
\sigma_{c,d,e}
}.
$$

Рабочая метрика положительного отклонения для будущего детектора:

$$
z_{+,c,d,e}(t)
=
\frac{
\left[
s_{c,d,e}(t)-F_{75,c,d,e}
\right]_+
}{
\sigma_{c,d,e}
},
$$

где

$$
F_{75,c,d,e}
=
Q_{0.75}^{w}\left(s_{c,d,e}\right).
$$

### H0-global

После локального удаления уровня, масштаба и учёта дискретизации форма распределения тихого спреда практически не зависит от монеты:

$$
H_{0,\mathrm{global}}:
\qquad
\mathcal L
\left(
r_{c,d,e}(t)
\mid\mathrm{quiet}
\right)
\approx
\mathcal P_d.
$$

Следствием этой гипотезы является возможность использовать одно глобальное значение $z_{\mathrm{det}}$ отдельно для `long` и `short`.

### H0-liquidity

Если глобальная гипотеза отвергается, проверяется более слабая гипотеза:

$$
H_{0,\mathrm{liq}}:
\qquad
\mathcal L
\left(
r_{c,d,e}(t)
\mid g(c),\mathrm{quiet}
\right)
\approx
\mathcal P_{g(c),d},
$$

где $g(c)$ — заранее зафиксированный класс ликвидности.

### H0-percentile

Если распределения $r$ не подобны даже внутри liquidity-класса, остаётся гипотеза автоматической per-coin нормировки:

$$
u_{c,d}(t)
=
F^0_{c,d}\left(r_{c,d}(t)\right),
$$

после которой для всех монет используется один процентильный порог:

$$
u_{c,d}(t)\geq p_{\mathrm{det}}.
$$

Это всё ещё универсальная методология: численные пороги монет не выбираются вручную.

---

## 3. Дизайн выборки

Пользователь до начала расчётов выбирает:

* 10 монет;
* несколько заранее обозначенных сегментов ликвидности;
* по 3 тихих интервала каждой монеты;
* продолжительность каждого интервала — ровно 12 часов.

Итого:

$$
N_{\mathrm{windows}}=10\times3=30.
$$

Направления `long` и `short` анализируются отдельно. Один интервал создаёт две временные серии, но остаётся одной календарной экспериментальной единицей.

Три интервала одной монеты должны по возможности относиться к разным тихим режимам или дням. Если несколько окон принадлежат одному непрерывному тихому режиму, им присваивается один `quiet_regime_id`, и они не считаются независимыми при бутстрепе.

---

## 4. Правила ручного выбора тихих интервалов

Ручной выбор допустим, поскольку эксперимент условен на состоянии `quiet`. Но интервалы выбираются до просмотра рассчитанных $r$, $z$, квантилей и результатов сравнения.

Тихий интервал должен удовлетворять следующим визуальным условиям:

* отсутствует продолжительный уход спреда на новый уровень;
* отсутствует выраженный аномальный эпизод;
* допустимы короткие дискретные прострелы;
* флор визуально существует и не совершает очевидный структурный переход;
* нет существенных дыр, рестартов или загрязнённых участков;
* интервал не выбран только потому, что его гистограмма выглядит особенно удобной.

Внутри заранее объявленной календарной области следует сохранять не только принятые окна, но и отклонённые кандидаты с причиной отклонения.

Liquidity-классы фиксируются до расчёта $z$ и не изменяются после просмотра результата.

---

## 5. Входной manifest

```text
quiet_id
quiet_regime_id
calendar_cluster_id
base_coin
direction
liquidity_class_pre
window_start_ts_utc
window_end_ts_utc
quiet_visual_verified
data_valid
contaminated
selection_notes
rejection_reason
```

Для primary используются только строки:

```text
quiet_visual_verified=true
data_valid=true
contaminated=false
```

После получения 30 валидных окон manifest замораживается. Любая замена интервала требует новой версии manifest и полного повторного прогона.

---

## 6. Разделение каждого 12-часового окна

Каждое окно делится хронологически без подбора точки разделения:

$$
Q_{c,e}^{\mathrm{cal}}
=
[t_0,t_0+6\mathrm{h}),
$$

$$
Q_{c,e}^{\mathrm{eval}}
=
[t_0+6\mathrm{h},t_0+12\mathrm{h}).
$$

По первой половине оцениваются:

$$
F_0,\qquad F_{0,25},\qquad F_{0,75},\qquad \sigma_0.
$$

На второй половине рассчитывается причинная метрика:

$$
r^{\mathrm{causal}}(t)
=
\frac{s(t)-F_0}{\sigma_0},
$$

$$
z_+^{\mathrm{causal}}(t)
=
\frac{[s(t)-F_{0,75}]_+}{\sigma_0}.
$$

Первая половина короче предполагаемого $H_{\mathrm{floor}}=12$ часов, поэтому согласно методологии используется целиком. Этот split не задаёт окончательное значение $H_{\mathrm{floor}}$ для стратегии.

Для диагностического разделения причин во второй половине отдельно рассчитываются:

$$
F_1
=
Q_{0.50}^{w}
\left(
s(t)\mid Q^{\mathrm{eval}}
\right),
$$

$$
\sigma_1
=
1.4826\,
\operatorname{wMAD}
\left(
s(t)\mid Q^{\mathrm{eval}}
\right),
$$

$$
r^{\mathrm{local}}(t)
=
\frac{s(t)-F_1}{\sigma_1}.
$$

Таким образом:

* $r^{\mathrm{causal}}$ проверяет перенос флора и масштаба вперёд по времени;
* $r^{\mathrm{local}}$ проверяет только подобие формы распределений после локальной нормировки.

---

## 7. Метрики одного окна

### 7.1 Высокие квантили

Для $r^{\mathrm{causal}}$, $r^{\mathrm{local}}$ и $z_+^{\mathrm{causal}}$:

$$
Q_{0.95},\qquad Q_{0.97},\qquad Q_{0.99}.
$$

P95, P97 и P99 являются primary tail metrics. Более высокие квантили допускаются только как диагностика при достаточном хвостовом support.

### 7.2 Ширина тихого распределения

$$
B_{50}
=
Q_{0.75}(r)-Q_{0.25}(r),
$$

$$
B_{90}
=
Q_{0.95}(r)-Q_{0.05}(r).
$$

$B_{50}$ характеризует основную ширину тихого режима, а $B_{90}$ — ширину с учётом хвостов.

### 7.3 Движение флора

Сдвиг между половинами окна:

$$
\Delta_F
=
\frac{F_1-F_0}{\sigma_0}.
$$

Изменение масштаба:

$$
R_\sigma
=
\frac{\sigma_1}{\sigma_0}.
$$

Для симметричного представления используется:

$$
L_\sigma=\log R_\sigma.
$$

Интерпретация:

* $\Delta_F\approx0$ — флор переносится вперёд;
* $L_\sigma\approx0$ — масштаб тихого режима сохраняется;
* стабильные хвосты $r^{\mathrm{local}}$ при нестабильных $r^{\mathrm{causal}}$ означают проблему движения флора, а не формы распределения.

### 7.4 Дискретизация

Для каждого окна рассчитываются:

```text
tick_size_okx
tick_size_bybit
effective_spread_resolution
resolution_ratio_kappa
max_atom_mass
n_occupied_levels
```

Нормированное разрешение:

$$
\kappa
=
\operatorname{median}_t
\left[
\frac{\delta_s(t)}{\sigma_0}
\right],
$$

где $\delta_s(t)$ — локальный эффективный шаг исполнимого спреда, рассчитанный из фактической формулы спреда и tick size обеих бирж.

Максимальная масса одного уровня:

$$
A_{\max}
=
\max_x
P\left(r=x\right).
$$

Если $\operatorname{wMAD}=0$, основная $z$-нормировка признаётся недоступной:

```text
normalization_status=tick_resolution_limited
```

Подстановка $\sigma_{\mathrm{eff}}=\max(\sigma_{\mathrm{MAD}},\delta_s)$ допускается только как отдельный sensitivity arm и не заменяет primary-расчёт.

---

## 8. Поправка на дискретизацию

Сравнение проводится в двух представлениях.

### 8.1 Observed arm

Сравниваются фактические нормированные распределения. Этот arm отвечает на прикладной вопрос:

> Можно ли использовать одно численное значение $z_{\mathrm{det}}$ на фактических L1-ценах?

### 8.2 Resolution-aware arm

Распределения сравниваются после согласования разрешения либо через дискретные хвостовые вероятности.

Для значения $x$ процентильный ранг одного атома задаётся интервалом:

$$
\left[
F(x^-),F(x)
\right].
$$

Дискретная верхняя хвостовая вероятность:

$$
p^+(x)=P_0(r\geq x).
$$

Для статистической диагностики допускается randomized PIT:

$$
u^{\mathrm{rand}}(x)
=
F(x^-)
+
V\left[F(x)-F(x^-)\right],
\qquad
V\sim U(0,1).
$$

Randomized PIT не переносится в торговое решение.

Если observed arm отвергает глобальное подобие, а resolution-aware arm нет, фиксируется вывод:

```text
latent_shape_not_rejected
global_numeric_z_rejected_by_tick_resolution
```

Это означает переход к дискретному процентильному порогу, а не автоматический отказ от всей нормировки.

---

## 9. Сравнение внутри одной монеты

Для каждого $p\in{0.95,0.97,0.99}$ выполняется cross-application трёх окон.

Порог, полученный на окне $e_1$:

$$
q_{c,e_1}(p)
=
Q_p\left(r_{c,e_1}\right),
$$

проверяется на другом окне $e_2$ той же монеты.

Из-за дискретизации на проверочном окне рассчитывается интервал частоты превышения:

$$
\Pi_{c,e_2}(p)
=
\left[
P(r>q_{c,e_1}(p)),
P(r\geq q_{c,e_1}(p))
\right].
$$

Ожидаемая частота:

$$
\alpha_p=1-p.
$$

Пилотный интервал практической эквивалентности:

$$
\mathcal A_p
=
\left[
\frac{\alpha_p}{2},
2\alpha_p
\right].
$$

Перенос между двумя окнами считается практически допустимым, если

$$
\Pi_{c,e_2}(p)\cap\mathcal A_p\neq\varnothing.
$$

Коэффициент 2 является заранее фиксируемой инженерной точностью пилота, а не универсальным законом. В большом подтверждающем эксперименте он должен быть связан с допустимой частотой ложных сигналов стратегии.

Для каждой монеты и каждого $p$ формируется матрица из шести направленных переносов между тремя окнами.

---

## 10. Глобальная leave-one-coin-out проверка

Для каждой исключённой монеты $c$ глобальный порог строится только по остальным девяти монетам:

$$
q_{-c}(p)
=
Q_p
\left(
r_{c'\neq c}
\right).
$$

Порог проверяется отдельно на трёх окнах исключённой монеты.

Это исключает ситуацию, когда BTC или конкретный альт сам участвует в построении порога, переносимость которого на него проверяется.

Аналогично выполняется liquidity leave-one-coin-out:

$$
q_{g,-c}(p)
=
Q_p
\left(
r_{c'\in g(c),\,c'\neq c}
\right).
$$

Если в классе остаётся меньше двух других монет, результат внутри класса считается только описательным.

---

## 11. Расстояния между распределениями

Основная описательная дистанция:

$$
D_{ij}^{W}
=
W_1
\left(
\widehat{\mathcal P}_i,
\widehat{\mathcal P}_j
\right).
$$

Отдельно рассчитываются:

* расстояния между окнами одной монеты;
* расстояния между монетами одного liquidity-класса;
* расстояния между liquidity-классами;
* глобальные расстояния между всеми монетами;
* raw и resolution-matched варианты.

KS p-value не используется как основной критерий из-за зависимости тиков и дискретных атомов.

---

## 12. Разложение вариабельности

Для каждой оконной метрики $Y_{c,e}$:

$$
Y_{c,e}
=
\mu
+
a_{g(c)}
+
b_c
+
\varepsilon_{c,e},
$$

где:

* $a_{g(c)}$ — эффект liquidity-класса;
* $b_c$ — остаточный эффект монеты;
* $\varepsilon_{c,e}$ — различия между тихими окнами одной монеты.

Доля вариации, объясняемая монетой:

$$
\operatorname{ICC}_{\mathrm{coin}}
=
\frac{
\sigma_{\mathrm{coin}}^2
}{
\sigma_{\mathrm{coin}}^2+
\sigma_{\mathrm{window}}^2
}.
$$

После учёта liquidity-класса отдельно оценивается:

$$
\operatorname{ICC}_{\mathrm{coin}\mid\mathrm{liq}}.
$$

Интерпретация:

* высокий $\operatorname{ICC}_{\mathrm{coin}}$ — форма систематически зависит от монеты;
* низкий $\operatorname{ICC}_{\mathrm{coin}}$ — различия между монетами не превышают естественную смену тихих окон;
* снижение ICC после добавления liquidity-класса поддерживает кластерную гипотезу.

---

## 13. Независимость и бутстреп

Тики не считаются независимыми наблюдениями.

Основная независимая единица:

```text
quiet_regime_id
```

Если окна разных монет совпадают по UTC и могут иметь общий рыночный контекст, используется:

```text
calendar_cluster_id
```

Доверительные интервалы строятся кластерным бутстрепом:

1. ресэмплируются календарные или quiet-regime кластеры;
2. внутри них сохраняются все зависимые тики и направления;
3. для cross-market вывода дополнительно ресэмплируются монеты.

Запрещён обычный bootstrap отдельных строк или тиков.

---

## 14. Критерии результата пилота

Пилот имеет четыре допустимых вердикта.

### 14.1 `REJECT_GLOBAL`

Глобальная гипотеза отвергается, если после resolution-aware поправки одновременно выполняются два условия.

Первое условие: для как минимум двух метрик из

$$
Q_{0.95},\quad Q_{0.97},\quad Q_{0.99}
$$

нижняя граница 95% кластерного bootstrap CI удовлетворяет

$$
\operatorname{LCB}_{0.95}
\left(
\operatorname{ICC}_{\mathrm{coin}}
\right)
>0.5.
$$

Это означает, что различия между монетами объясняют больше вариации, чем смена тихих окон внутри монеты.

Второе условие: в глобальной leave-one-coin-out проверке существуют как минимум две монеты из разных liquidity-классов, для которых:

* нарушение наблюдается на всех трёх окнах;
* нарушение направлено в одну сторону;
* оно присутствует минимум на двух уровнях из P95/P97/P99;
* tie-aware интервал превышения не пересекает

$$
\left[
\frac{1-p}{2},
2(1-p)
\right].
$$

После `REJECT_GLOBAL` следующий эксперимент проектируется для одного заранее определённого liquidity-класса.

### 14.2 `REJECT_GLOBAL_NUMERIC_Z_ONLY`

Observed arm отвергает глобальный порог, но resolution-aware arm не отвергает подобие формы.

Вывод:

* общий численный $z_{\mathrm{det}}$ непереносим из-за tick size;
* гипотеза общей базовой формы не отвергнута;
* следующий кандидат — универсальный дискретный процентиль $p_{\mathrm{det}}$.

### 14.3 `ADVANCE_GLOBAL`

Глобальная гипотеза не считается доказанной, но переводится в расширенный эксперимент, если:

* условия `REJECT_GLOBAL` не выполнены;
* результат не определяется окнами с `tick_resolution_limited`;
* направление вывода одинаково для `long` и `short` либо различие направлений явно описано;
* bootstrap CI конечны;
* нет одной монеты, полностью определяющей pooled-квантили;
* внутри монет переносимость не хуже глобальной настолько, чтобы эксперимент терял смысл.

Следующий эксперимент использует большее число монет и независимых тихих режимов.

### 14.4 `INCONCLUSIVE`

Эксперимент признаётся неразрешающим, если:

* недостаточно независимых quiet regimes;
* интервалы загрязнены или пересекают дыры;
* слишком много окон имеют $\operatorname{wMAD}=0$;
* bootstrap CI не позволяют различить внутримонетную и межмонетную вариабельность;
* отдельные окна одной монеты настолько различны, что невозможно проверить межмонетную переносимость;
* raw и resolution-aware результаты расходятся, но влияние tick size не локализовано.

`INCONCLUSIVE` не трактуется как усиление H0.

---

## 15. Разбор причины возможного отвержения

При отвержении обязательно указывается механизм.

| Наблюдение                                                         | Интерпретация                                  |
| ------------------------------------------------------------------ | ---------------------------------------------- |
| Нестабилен $r^{\mathrm{causal}}$, но стабилен $r^{\mathrm{local}}$ | Движется флор или масштаб                      |
| Нестабилен и $r^{\mathrm{local}}$                                  | Различается форма тихого распределения         |
| Raw различается, resolution-aware совпадает                        | Причина в tick size                            |
| Весь рынок различается, внутри liquidity-класса совпадает          | Поддерживается кластерная нормировка           |
| Даже окна одной монеты не совпадают                                | Тихий режим требует дополнительного разделения |
| Per-coin percentile переносится, общий $z$ нет                     | Использовать автоматическую CDF монеты         |

---

## 16. Отчёт эксперимента

Исполнитель должен сформировать:

1. Замороженный manifest 30 окон.
2. Таблицу оконных статистик.
3. Таблицу P95/P97/P99 для всех представлений.
4. Матрицы внутримонетного cross-application.
5. Global и liquidity leave-one-coin-out calibration.
6. Raw и resolution-aware distance matrices.
7. Variance decomposition и ICC с cluster-bootstrap CI.
8. Таблицу дискретизации: $\kappa$, $A_{\max}$, число уровней, MAD-status.
9. HTML-атлас всех 30 окон.
10. Итоговый verdict по четырём допустимым категориям.

Минимальные графики:

* три ECDF одного цвета на каждую монету;
* QQ-графики окон одной монеты;
* QQ-графики между монетами;
* heatmap P95/P97/P99;
* heatmap leave-one-coin-out calibration;
* raw и resolution-aware distance matrices;
* $\kappa$ против ошибки хвостовой калибровки;
* liquidity против хвостовых квантилей;
* $\Delta_F$ и $L_\sigma$ по окнам;
* variance decomposition: class / coin / window.

---

## 17. Ограничения вывода

Эксперимент не доказывает:

* наличие торгового преимущества;
* возврат аномального спреда к флору;
* переносимость $W$, $O_{\min}$ и $I_{\min}$;
* прибыльность после fees и Trade_Lat;
* пригодность одного порога для живого бота;
* причинную природу отклонений.

Он проверяет только то, можно ли считать median/MAD-нормированный тихий спред сопоставимым между окнами и монетами на масштабе, важном для будущих P95–P99 порогов.

---

## 18. Handoff исполнителю

До кода исполнитель обязан:

1. Проверить и заморозить manifest.
2. Зафиксировать liquidity labels.
3. Зафиксировать формулу эффективного spread resolution.
4. Подтвердить временное взвешивание всех распределений.
5. Подтвердить отсутствие bootstrap по тикам.
6. Зафиксировать primary P95/P97/P99 и критерии verdict.
7. Не смотреть PnL и не менять параметры канонического симулятора.

После выпуска отчёта исполнитель останавливается. Любое изменение нормировки, liquidity-классов, окон или критериев считается новым экспериментом.
